# Preprocessing danych — Książki (Book-Crossing)

## Cel tego notebooka
Przygotowanie danych z datasetu Book-Crossing do trenowania modelu regresji liniowej i logistycznej.
Dataset zawiera oceny książek wystawione przez użytkowników w skali 0–10,
gdzie 0 oznacza brak oceny (implicit feedback) — te rekordy zostaną usunięte.

Pipeline jest analogiczny do preprocessingu filmów (02_preprocessing.ipynb) —
ten sam model, te same założenia, inny typ treści.

📄 Szczegółowe wyjaśnienie: docs/14_books_preprocessing.md

In [ ]:
import os
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import StandardScaler

os.chdir(os.path.expanduser('~/magisterka'))
print('Working directory:', os.getcwd())

## Wczytanie danych

Dataset Book-Crossing używa separatora `;` zamiast standardowego `,`.
Wczytujemy trzy pliki:
- `Ratings.csv` — oceny użytkowników (User-ID, ISBN, Rating)
- `Books.csv` — metadane książek (ISBN, tytuł, autor, rok, wydawca)
- `Users.csv` — dane demograficzne użytkowników (User-ID, Location, Age)

Parametr `encoding='latin-1'` jest konieczny bo pliki zawierają znaki spoza ASCII
(np. polskie lub francuskie litery w tytułach książek).

In [ ]:
ratings = pd.read_csv('data/books/Ratings.csv', sep=';', encoding='latin-1')
books   = pd.read_csv('data/books/Books.csv',   sep=';', encoding='latin-1', on_bad_lines='skip')
users   = pd.read_csv('data/books/Users.csv',   sep=';', encoding='latin-1')

print(f'Ratings: {ratings.shape}')
print(f'Books:   {books.shape}')
print(f'Users:   {users.shape}')
display(ratings.head(3))
display(books.head(3))
display(users.head(3))

## Ujednolicenie nazw kolumn

Nazwy kolumn w plikach Book-Crossing zawierają myślniki i wielkie litery.
Zamieniamy je na spójny format snake_case żeby kod był czytelny
i zgodny z konwencją używaną w notebooku filmowym.

In [ ]:
ratings.columns = ['userId', 'isbn', 'rating']
books.columns   = ['isbn', 'title', 'author', 'year', 'publisher',
                    'img_s', 'img_m', 'img_l']
users.columns   = ['userId', 'location', 'age']

# zostawiamy tylko potrzebne kolumny
books = books[['isbn', 'title', 'author', 'year', 'publisher']]

print('Kolumny ratings:', ratings.columns.tolist())
print('Kolumny books:  ', books.columns.tolist())
print('Kolumny users:  ', users.columns.tolist())

## Usunięcie ocen zerowych (implicit feedback)

W datasecie Book-Crossing ocena 0 oznacza że użytkownik nie wystawił
prawdziwej oceny — jedynie odnotowano że miał kontakt z książką.
Takie rekordy nie niosą informacji o preferencjach i zostają usunięte.

Z 1 149 780 rekordów zostaje ~433 671 prawdziwych ocen (1–10).

In [ ]:
print(f'Ocen przed filtrowaniem: {len(ratings)}')
print(f'Ocen zerowych (implicit): {(ratings["rating"] == 0).sum()}')

ratings = ratings[ratings['rating'] > 0].copy()

print(f'Ocen po filtrowaniu: {len(ratings)}')
print('\nRozkład ocen:')
print(ratings['rating'].value_counts().sort_index())

## Czyszczenie danych użytkowników

Kolumna `age` zawiera brakujące wartości i nierealistyczne liczby
(np. wiek 0 lub 244). Wypełniamy braki medianą i przycinamy do zakresu 5–90.

In [ ]:
print(f'Brakujące wieki: {users["age"].isna().sum()}')
print(f'Wiek min/max przed: {users["age"].min()} / {users["age"].max()}')

median_age = users['age'].median()
users['age'] = users['age'].fillna(median_age)
users['age'] = users['age'].clip(5, 90)

print(f'Mediana wieku: {median_age}')
print(f'Wiek min/max po: {users["age"].min()} / {users["age"].max()}')

## Czyszczenie danych książek

Kolumna `year` zawiera błędne wartości (np. '0', 'DK', lata > 2024).
Konwertujemy na liczby, niepoprawne wartości zastępujemy medianą.

In [ ]:
books['year'] = pd.to_numeric(books['year'], errors='coerce')
books['year'] = books['year'].where((books['year'] >= 1800) & (books['year'] <= 2024))

median_year = books['year'].median()
books['year'] = books['year'].fillna(median_year)

print(f'Mediana roku wydania: {median_year}')
print(f'Rok min/max: {books["year"].min()} / {books["year"].max()}')
display(books.head(3))

## Filtrowanie aktywnych użytkowników i popularnych książek

Użytkownicy którzy ocenili mniej niż 5 książek nie dają modelowi wystarczającego
sygnału. Książki z mniej niż 5 ocenami są zbyt rzadkie żeby model mógł się
na nich uczyć. Filtrowanie poprawia jakość modelu i redukuje szum w danych.

Próg 5 to kompromis między jakością a rozmiarem datasetu.

In [ ]:
MIN_USER_RATINGS = 5
MIN_BOOK_RATINGS = 5

user_counts = ratings['userId'].value_counts()
book_counts = ratings['isbn'].value_counts()

active_users = user_counts[user_counts >= MIN_USER_RATINGS].index
popular_books = book_counts[book_counts >= MIN_BOOK_RATINGS].index

ratings_filtered = ratings[
    ratings['userId'].isin(active_users) &
    ratings['isbn'].isin(popular_books)
].copy()

print(f'Ocen przed filtrowaniem:  {len(ratings)}')
print(f'Ocen po filtrowaniu:      {len(ratings_filtered)}')
print(f'Aktywnych użytkowników:   {ratings_filtered["userId"].nunique()}')
print(f'Popularnych książek:      {ratings_filtered["isbn"].nunique()}')

## Łączenie tabel (merge)

Łączymy trzy tabele w jedną — tak samo jak w notebooku filmowym.
Każdy rekord będzie zawierał ocenę + informacje o książce + dane użytkownika.

Używamy `inner join` — zachowujemy tylko rekordy które mają dopasowanie
we wszystkich trzech tabelach.

In [ ]:
df = ratings_filtered.merge(books, on='isbn', how='inner')
df = df.merge(users, on='userId', how='inner')

print(f'Rozmiar po merge: {df.shape}')
print(f'Kolumny: {df.columns.tolist()}')
display(df.head(3))

## Statystyki agregowane

Obliczamy średnią ocenę każdego użytkownika i każdej książki
oraz liczbę ocen dla każdej książki.
Te wartości będą cechami modelu — pomagają mu uwzględnić
ogólne tendencje (np. user który zawsze daje wysokie oceny
albo książka która jest powszechnie lubiana).

In [ ]:
user_avg = df.groupby('userId')['rating'].mean().rename('user_avg_rating')
book_avg = df.groupby('isbn')['rating'].mean().rename('book_avg_rating')
book_count = df.groupby('isbn')['rating'].count().rename('book_rating_count')

df = df.merge(user_avg, on='userId')
df = df.merge(book_avg, on='isbn')
df = df.merge(book_count, on='isbn')

print('Statystyki user_avg_rating:')
print(df['user_avg_rating'].describe().round(3))
print('\nStatystyki book_avg_rating:')
print(df['book_avg_rating'].describe().round(3))

## Budowanie zbioru cech (feature engineering)

Wybieramy cechy które model będzie wykorzystywał do predykcji.
W przeciwieństwie do filmów, książki nie mają gatunków w tym datasecie —
używamy cech demograficznych użytkownika i statystyk książki.

Cechy:
- `age` — wiek użytkownika (liczba)
- `user_avg_rating` — średnia ocen użytkownika (jego ogólna hojność)
- `book_avg_rating` — średnia ocen książki (jej ogólna popularność)
- `book_rating_count` — liczba ocen książki (jej popularność)
- `year` — rok wydania książki

Zmienna docelowa (TARGET): `rating` — ocena którą użytkownik wystawił książce.

In [ ]:
FEATURE_COLS = [
    'age',
    'user_avg_rating',
    'book_avg_rating',
    'book_rating_count',
    'year',
]
TARGET_COL = 'rating'

df_model = df[FEATURE_COLS + [TARGET_COL, 'userId', 'isbn', 'title']].copy()
df_model = df_model.dropna()

print(f'Rozmiar df_model: {df_model.shape}')
print(f'Cechy: {FEATURE_COLS}')
display(df_model.head(3))

## Per-user temporal split

Dataset Book-Crossing nie zawiera kolumny timestamp — nie możemy posortować
ocen chronologicznie. Używamy losowego podziału per-user:
dla każdego użytkownika 80% jego ocen trafia do treningu, 20% do testu.
Shuffle wewnątrz każdego użytkownika zapewnia losowość.

Jest to mniej idealne niż temporal split (użyty dla filmów),
ale uczciwe — każdy użytkownik ma reprezentację w obu zbiorach.

In [ ]:
from sklearn.utils import shuffle

train_rows = []
test_rows  = []

for user_id, group in df_model.groupby('userId'):
    group = shuffle(group, random_state=42)
    split_idx = int(len(group) * 0.8)
    train_rows.append(group.iloc[:split_idx])
    test_rows.append(group.iloc[split_idx:])

train_df = pd.concat(train_rows).reset_index(drop=True)
test_df  = pd.concat(test_rows).reset_index(drop=True)

X_train = train_df[FEATURE_COLS]
y_train = train_df[TARGET_COL]
X_test  = test_df[FEATURE_COLS]
y_test  = test_df[TARGET_COL]

print(f'Train: {X_train.shape}')
print(f'Test:  {X_test.shape}')
print(f'Użytkowników w train: {train_df["userId"].nunique()}')
print(f'Użytkowników w test:  {test_df["userId"].nunique()}')

## Skalowanie cech (StandardScaler)

Cechy mają bardzo różne zakresy: `age` → 5–90, `book_rating_count` → 5–2500.
Standaryzacja przekształca każdą cechę do średniej=0 i odchylenia=1.
Scaler trenujemy TYLKO na danych treningowych — dane testowe
transformujemy tym samym scalerem (bez ponownego dopasowania).
Zapobiega to wyciekowi informacji z danych testowych.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# konwersja z powrotem na DataFrame z nazwami kolumn
X_train_scaled = pd.DataFrame(X_train_scaled, columns=FEATURE_COLS)
X_test_scaled  = pd.DataFrame(X_test_scaled,  columns=FEATURE_COLS)

print('Skalowanie zakończone.')
print('Średnie po skalowaniu (powinny być ~0):')
print(X_train_scaled.mean().round(4))

## Zapis danych i artefaktów

Zapisujemy przetworzone dane i artefakty do późniejszego użycia:
- `df_model_books.csv` — pełna tabela z cechami i metadanymi
- `X_train_books.csv`, `X_test_books.csv` — zbiory cech
- `y_train_books.csv`, `y_test_books.csv` — zmienne docelowe
- `scaler_books.pkl` — scaler do użycia w backendzie
- `feature_cols_books.pkl` — lista cech (kolejność musi być zachowana)

In [ ]:
os.makedirs('data/books_processed', exist_ok=True)
os.makedirs('backend/model_books', exist_ok=True)

df_model.to_csv('data/books_processed/df_model_books.csv', index=False)
X_train_scaled.to_csv('data/books_processed/X_train_books.csv', index=False)
X_test_scaled.to_csv('data/books_processed/X_test_books.csv', index=False)
y_train.to_csv('data/books_processed/y_train_books.csv', index=False)
y_test.to_csv('data/books_processed/y_test_books.csv', index=False)

joblib.dump(scaler,       'backend/model_books/scaler_books.pkl')
joblib.dump(FEATURE_COLS, 'backend/model_books/feature_cols_books.pkl')

print('Zapisano wszystkie pliki.')
print(f'df_model_books: {df_model.shape}')
print(f'X_train: {X_train_scaled.shape}, X_test: {X_test_scaled.shape}')